<a href="https://colab.research.google.com/github/piriyadharshiniN/24ADI006-DL-Lab/blob/main/DL_Exp_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

print("Name: Piriyadharshini. N\nRoll no.: 24BAD086\nExperiment: 9")

import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import AutoTokenizer

# Load IMDb dataset
dataset = load_dataset("stanfordnlp/imdb")

# Display dataset information
print("Training Samples:", len(dataset['train']))
print("Testing Samples:", len(dataset['test']))

# Display sample reviews
print("\nSample Review:")
print(dataset['train'][0]['text'])

print("\nSample Label:")
print(dataset['train'][0]['label'])

# Check label distribution
print("\nLabel Distribution:")
print(pd.Series(dataset['train']['label']).value_counts())

# Load pretrained BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased"
)

# Select smaller dataset for faster training
train_data = dataset['train'].shuffle(seed=42).select(range(2000))
test_data = dataset['test'].shuffle(seed=42).select(range(500))

# Tokenization function
def tokenize_data(examples):

    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

# Tokenize training and testing data
train_data = train_data.map(tokenize_data, batched=True)
test_data = test_data.map(tokenize_data, batched=True)

# Set required format
train_data.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'label']
)

test_data.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'label']
)

print("\nTokenization Completed!")
print("Input IDs Shape:", train_data[0]['input_ids'].shape)
print("Attention Mask Shape:", train_data[0]['attention_mask'].shape)

Name: Piriyadharshini. N
Roll no.: 24BAD086
Experiment: 9


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (8 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Tokenization Completed!
Input IDs Shape: torch.Size([128])
Attention Mask Shape: torch.Size([128])


In [5]:
# Name:Piriyadharshini N
# Roll No: 24BAD086
# Experiment No: 9

import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Select device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device:", device)

# Rename label column to labels
train_data = train_data.rename_column("label", "labels")
test_data = test_data.rename_column("label", "labels")

# Load pretrained BERT model
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=2
)

model.to(device)

# Prepare DataLoaders
train_loader = DataLoader(
    train_data,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=8
)

# Define optimizer
optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

# Train the model
epochs = 2

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch in train_loader:

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        optimizer.zero_grad()

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}"
    )


# Evaluate the model
model.eval()

predictions = []
actual_labels = []

with torch.no_grad():

    for batch in test_loader:

        labels = batch['labels'].to(device)

        inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != 'labels'
        }

        outputs = model(**inputs)

        preds = torch.argmax(
            outputs.logits,
            dim=1
        )

        predictions.extend(preds.cpu().tolist())
        actual_labels.extend(labels.cpu().tolist())


# Calculate evaluation metrics
print("\nClassification Results")

print("Accuracy:",
      accuracy_score(actual_labels, predictions))

print("Precision:",
      precision_score(actual_labels, predictions,
                      zero_division=0))

print("Recall:",
      recall_score(actual_labels, predictions,
                   zero_division=0))

print("F1 Score:",
      f1_score(actual_labels, predictions,
               zero_division=0))


# Save model and tokenizer
model.save_pretrained("bert_sentiment_model")
tokenizer.save_pretrained("bert_sentiment_model")

print("\nModel Saved Successfully!")

Using Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/2, Loss: 0.4705
Epoch 2/2, Loss: 0.2381

Classification Results
Accuracy: 0.834
Precision: 0.8755760368663594
Recall: 0.7723577235772358
F1 Score: 0.8207343412526998


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model Saved Successfully!


In [6]:
print("Name: Piriyadharshini. N\nRoll no.: 24BAD086\nExperiment: 9")


from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load T5 tokenizer
t5_tokenizer = T5Tokenizer.from_pretrained(
    "google-t5/t5-small"
)

# Load pretrained T5 model
t5_model = T5ForConditionalGeneration.from_pretrained(
    "google-t5/t5-small"
)

# Prepare input text
input_text = """
Artificial Intelligence is transforming the world.
It is used in healthcare, education, transportation
and many other industries.
It helps automate tasks and improve decision making.
"""

# Add summarization task prefix
text = "summarize: " + input_text

# Tokenize input
inputs = t5_tokenizer(
    text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

# Generate summary
outputs = t5_model.generate(
    inputs['input_ids'],
    max_length=50,
    num_beams=4,
    early_stopping=True
)

# Decode generated output
summary = t5_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# Display results
print("Original Text:")
print(input_text)

print("\nGenerated Summary:")
print(summary)

# Save generated output
with open("generated_summary.txt", "w") as file:
    file.write(summary)

print("\nOutput Saved Successfully!")

Name: Piriyadharshini. N
Roll no.: 24BAD086
Experiment: 9


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Original Text:

Artificial Intelligence is transforming the world.
It is used in healthcare, education, transportation
and many other industries.
It helps automate tasks and improve decision making.


Generated Summary:
artificial intelligence is used in healthcare, education, transportation and many other industries.

Output Saved Successfully!


In [7]:
print("Name: Piriyadharshini. N\nRoll no.: 24BAD086\nExperiment: 9")

# Test BERT with sample reviews
sample_reviews = [
    "This movie was excellent and amazing!",
    "The movie was boring and disappointing."
]

# Predict sentiment
model.eval()

for review in sample_reviews:

    # Tokenize input
    inputs = tokenizer(
        review,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Predict using BERT
    with torch.no_grad():

        outputs = model(**inputs)

        prediction = torch.argmax(
            outputs.logits,
            dim=1
        ).item()

    label = "Positive" if prediction == 1 else "Negative"

    print("\nReview:", review)
    print("Predicted Sentiment:", label)


# Test T5 with different input
sample_text = """
Machine learning is a branch of artificial intelligence.
It enables computers to learn from data and make predictions.
"""

# Prepare input for T5
text = "summarize: " + sample_text

inputs = t5_tokenizer(
    text,
    return_tensors="pt"
)

# Generate summary
outputs = t5_model.generate(
    inputs['input_ids'],
    max_length=40,
    num_beams=4
)

generated_text = t5_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# Display generated output
print("\nOriginal Text:")
print(sample_text)

print("\nGenerated Text:")
print(generated_text)

Name: Piriyadharshini. N
Roll no.: 24BAD086
Experiment: 9

Review: This movie was excellent and amazing!
Predicted Sentiment: Positive

Review: The movie was boring and disappointing.
Predicted Sentiment: Negative

Original Text:

Machine learning is a branch of artificial intelligence.
It enables computers to learn from data and make predictions.


Generated Text:
machine learning is a branch of artificial intelligence. it enables computers to learn from data and make predictions.
